# Vaccine Sense — Aplicação 19

## Treinamento do modelo

Treina o modelo que classifica a condição da carga em **TRANSPORTE_OK** ou
**CARGA_EM_PERIGO**, a partir do dataset rotulado na Aplicação 18.

No final, o `.pkl` vai para a pasta `app/` e a aplicação Python passa a
classificar cada medição nova que chega da caixa.


## 1. Pacotes


In [ ]:
!pip -q install pandas matplotlib scikit-learn joblib


## 2. Imports


In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)


## 3. Abrir o CSV da Aplicação 18

Selecione o `vaccinesense_dataset.csv` que você baixou no final da Aplicação 18.

Se ainda não coletou, use o exemplo em `dataset_gerado/` do app18.


In [ ]:
from google.colab import files

arquivos = files.upload()
ARQUIVO = list(arquivos.keys())[0]


In [ ]:
df = pd.read_csv(ARQUIVO, parse_dates=["timestamp"])
df = df.set_index("timestamp").sort_index()

print(f"{len(df)} medições em {df.rodada.nunique()} rodadas")
df.head()


## 4. Definir a pergunta do modelo

O dataset tem três situações, mas o modelo responde a uma pergunta binária:
**a carga está em perigo?**

`AMBIENTE_HOSTIL` entra como **0**. A caixa está fechada e a carga está
preservada — é um ambiente quente, não é uma carga em perigo.

São justamente essas rodadas que impedem o modelo de aprender o atalho
*"temperatura interna subindo é perigo"* e alarmar em todo dia quente.


In [ ]:
df["situacao"] = df["situacao"].str.strip().str.upper()
df["alvo"] = (df["situacao"] == "CARGA_EM_PERIGO").astype(int)

display(df["situacao"].value_counts())
print()
print("Alvo binário:")
display(df["alvo"].value_counts().rename({0: "não-perigo", 1: "perigo"}))


## 5. Separar as features

Entram somente as **seis medições** dos sensores. Ficam de fora:

| Coluna | Por que não é feature |
|---|---|
| `id` | contador: separaria as classes pela **ordem da coleta** |
| `timestamp` | a mesma coisa, em outra forma |
| `tempoForaDaFaixa` | derivado da temperatura interna, e só cresce |
| `device` | metadado do experimento |
| `rodada` | metadado — mas guardamos, porque divide treino e teste |
| `situacao` / `alvo` | é a resposta, não a pergunta |


In [ ]:
FEATURES = [
    "tempInterna",
    "tempExterna",
    "umidade",
    "luz",
    "criticidade",
    "distancia",
]

X = df[FEATURES]
y = df["alvo"]
grupos = df["rodada"]

display(X.head())


## 6. Dividir treino e teste **por rodada**

Aqui está a decisão mais importante do notebook.

As medições chegam a **1 Hz**: duas leituras vizinhas são quase idênticas,
porque a caixa não muda em um segundo.

Se dividirmos aleatoriamente — ou estratificando por rodada, que espalha cada
rodada dos dois lados — praticamente **a mesma leitura** fica no treino e no
teste. O modelo acerta porque já viu aquele ponto, não porque aprendeu.

A divisão precisa respeitar a estrutura da coleta: **uma rodada inteira está
no treino ou está no teste, nunca nos dois**. É o que o `GroupShuffleSplit`
faz, agrupando por `rodada`.

> É por isso que o protocolo do app18 pede **duas rodadas para cada condição**.
> Com o par, uma fica no treino e ensina a outra. Se a condição existisse em
> uma rodada só e ela caísse no teste, o modelo erraria tudo ali sem ter como
> aprender.


In [ ]:
divisor = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
indices_treino, indices_teste = next(divisor.split(X, y, groups=grupos))

X_treino, X_teste = X.iloc[indices_treino], X.iloc[indices_teste]
y_treino, y_teste = y.iloc[indices_treino], y.iloc[indices_teste]
grupos_treino = grupos.iloc[indices_treino]

print("Rodadas no treino:", sorted(grupos.iloc[indices_treino].unique()))
print("Rodadas no teste: ", sorted(grupos.iloc[indices_teste].unique()))
print(f"\nTreino: {len(X_treino)} medições")
print(f"Teste:  {len(X_teste)} medições")


### Quanto o split errado infla a métrica

Vale medir a diferença uma vez, para não restar dúvida.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

modelo_teste = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=50, random_state=42)),
])

aleatorio = cross_val_score(
    modelo_teste, X, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="accuracy").mean()

por_rodada = cross_val_score(
    modelo_teste, X, y, groups=grupos,
    cv=GroupShuffleSplit(n_splits=8, test_size=0.25, random_state=42),
    scoring="accuracy").mean()

print(f"Split aleatório: {aleatorio:.3f}   <- parece ótimo, e é ilusão")
print(f"Split por rodada: {por_rodada:.3f}   <- o desempenho real")


## 7. Procurar o melhor modelo

O `GridSearchCV` treina vários modelos com vários hiperparâmetros e escolhe a
melhor combinação pelo **F1** da classe `CARGA_EM_PERIGO`.

Dois cuidados específicos deste projeto:

- a validação cruzada também agrupa por rodada (`GroupKFold`), senão o mesmo
  vazamento voltaria por dentro da busca;
- só entram modelos que o **m2cgen** consegue converter em C++, porque na
  Aplicação 21 este modelo vai rodar dentro do ESP32.


In [ ]:
print("Pipeline com múltiplos modelos usando GridSearchCV")

# Pipeline base com um lugar reservado para o classificador
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier())  # será substituído pelo GridSearchCV
])

# Validação cruzada agrupada: nenhuma rodada nos dois lados
cv = GroupKFold(n_splits=4)

# Grade de parâmetros para múltiplos modelos
param_grid_multi = [
    # 1. Árvore de decisão
    {
        "clf": [DecisionTreeClassifier(random_state=42)],
        "clf__max_depth": [3, 5, 8, None],
        "clf__min_samples_leaf": [1, 5, 20],
    },

    # 2. Random Forest
    {
        "clf": [RandomForestClassifier(random_state=42, n_jobs=-1)],
        "clf__n_estimators": [10, 50, 100],
        "clf__max_depth": [5, 10, None],
        "clf__min_samples_leaf": [1, 2],
    },

    # 3. Regressão logística
    {
        "clf": [LogisticRegression(max_iter=2000, random_state=42)],
        "clf__C": [0.1, 1, 10],
    },
]

print("Início do treinamento")

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid_multi,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_treino, y_treino, groups=grupos_treino)
print("Fim do treinamento")


## 8. Resultado do GridSearchCV (melhor modelo)


In [ ]:
melhor_modelo = grid_search.best_estimator_.named_steps["clf"]
melhor_nome = type(melhor_modelo).__name__

print(f"Melhor modelo: {melhor_nome}")
print(f"Melhor F1 na validação: {grid_search.best_score_:.4f}")
print("\nMelhores hiperparâmetros:")
for parametro, valor in grid_search.best_params_.items():
    if parametro.startswith("clf__"):
        print(f"  --> {parametro.replace('clf__', '')}: {valor}")

# As 5 melhores combinações
resultados = pd.DataFrame(grid_search.cv_results_)
display(
    resultados[["rank_test_score", "mean_fit_time", "mean_test_score", "params"]]
    .sort_values(by="mean_test_score", ascending=False)
    .head(5)
)


### O que os dados ensinam: uma árvore para ler

O vencedor do GridSearchCV pode ser uma floresta com 100 árvores — impossível
de ler. Então treinamos **uma árvore rasa só para explicar**: ela não é o
modelo que vai para a aplicação, é a lente para enxergar a regra que está nos
dados.

Olhe três coisas no desenho:

1. **o primeiro corte** — deve ser a `luz`, separando tampa aberta de fechada;
2. **os cortes de `tempInterna`** — são os limites da faixa segura;
3. **se aparecer `criticidade`** — o modelo descobriu sozinho que a faixa muda
   conforme o tipo da carga. É o corte mais interessante da árvore.


In [ ]:
# Uma árvore rasa, só para enxergar a regra.
# Aumente max_depth para 5 e veja a criticidade aparecer mais vezes.
arvore = DecisionTreeClassifier(max_depth=4, random_state=42)
arvore.fit(X_treino, y_treino)

plt.figure(figsize=(18, 8))
plot_tree(arvore, feature_names=FEATURES,
          class_names=["TRANSPORTE_OK", "CARGA_EM_PERIGO"],
          filled=True, rounded=True, fontsize=8)
plt.show()

# a mesma árvore em texto, que copia direto para o slide
print(export_text(arvore, feature_names=FEATURES,
                  class_names=["TRANSPORTE_OK", "CARGA_EM_PERIGO"]))


## 9. Testar e interpretar

As rodadas de teste nunca foram vistas no treinamento.


In [ ]:
predicoes = grid_search.predict(X_teste)

print(classification_report(
    y_teste, predicoes,
    target_names=["TRANSPORTE_OK", "CARGA_EM_PERIGO"]))

ConfusionMatrixDisplay(
    confusion_matrix(y_teste, predicoes),
    display_labels=["TRANSPORTE_OK", "CARGA_EM_PERIGO"],
).plot(cmap="Blues")
plt.show()


### Comparando com o jeito ingênuo

Antes do modelo, a forma óbvia de detectar tampa aberta seria um limiar de luz.
Vale medir o quanto o modelo é melhor que isso.

> **Calibre o limiar com a SUA caixa.** O valor de luz com a tampa aberta
> depende do tamanho da caixa, da posição do LDR e da iluminação da sala.


In [ ]:
LIMIAR_LUZ = 1000

baseline = (X_teste["luz"] > LIMIAR_LUZ).astype(int)

print(f"Modelo:   F1 = {f1_score(y_teste, predicoes):.3f}")
print(f"Limiar:   F1 = {f1_score(y_teste, baseline):.3f}")
print()
print("=== Regra por limiar de luz ===")
print(classification_report(
    y_teste, baseline,
    target_names=["TRANSPORTE_OK", "CARGA_EM_PERIGO"]))


O limiar de luz acerta as rodadas de tampa aberta e **erra todas** as de
temperatura fora da faixa. O modelo pega as duas famílias — é essa a diferença.

Onde o modelo ainda erra costuma ser a **fronteira**: carga padrão perto de
8 °C e carga crítica perto de 6 °C. É o erro certo, no lugar certo: ali o
limite físico é uma região, não uma linha.


## 10. Fazer um `.predict()`

É exatamente esta chamada que a aplicação Python fará com a última medição
recebida da caixa.


In [ ]:
uma_medicao = pd.DataFrame([{
    "tempInterna": 8.8,
    "tempExterna": 24.1,
    "umidade":     71.0,
    "luz":         62,
    "criticidade": 30,
    "distancia":   14.2,
}])[FEATURES]

predicao = grid_search.predict(uma_medicao)[0]

display(uma_medicao)
print("CARGA_EM_PERIGO" if predicao == 1 else "TRANSPORTE_OK")


## 11. Treinar com todos os dados e salvar

A avaliação acima já foi feita: sabemos como o modelo se comporta em rodadas
que ele nunca viu.

Agora repetimos o treinamento com **todas** as medições, usando os
hiperparâmetros vencedores. Quanto mais dados, melhor o modelo final — e é
esse que vai para a aplicação.


In [ ]:
MODELO_ARQUIVO = "modelo_vaccinesense.pkl"

modelo_final = grid_search.best_estimator_
modelo_final.fit(X, y)

joblib.dump(modelo_final, MODELO_ARQUIVO)
print(f"Modelo salvo: {MODELO_ARQUIVO} ({melhor_nome})")

# conferência: carrega e prevê, igual a aplicação vai fazer
modelo = joblib.load(MODELO_ARQUIVO)
print("Teste de carregamento:",
      "CARGA_EM_PERIGO" if modelo.predict(uma_medicao)[0] == 1 else "TRANSPORTE_OK")


In [ ]:
files.download(MODELO_ARQUIVO)


---

## Próximo passo

Coloque o `modelo_vaccinesense.pkl` na pasta `app/` da Aplicação 19 e rode:

```bash
python appConsoleVaccineSense.py
```

A aplicação lê a última medição da caixa no InfluxDB e classifica em tempo
real. Na Aplicação 21, este mesmo modelo vira C++ e roda dentro do ESP32.
